# Certificate Transparency Pattern Analysis

Explore synthetic certificate records to find unusual fingerprint and subject-alternative-name reuse.

**Safety and scope:** This notebook uses deterministic synthetic data and makes no network requests. Its results are analytical leads, not attribution or identity claims.

## Goal

Surface certificate clusters that may connect infrastructure while preserving uncertainty and context.


## Setup

The workflow runs offline with NumPy and Pandas. Parameters and source-like fields are visible so the analysis can be reviewed and rerun.

### Key Assumptions

- All records are synthetic and contain no real people or infrastructure.
- Scores prioritize review; they do not prove ownership, intent, identity, or location.
- Real use requires documented authority, provenance, source terms, and retention limits.


In [1]:
import numpy as np
import pandas as pd

SEED = 88
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)
pd.set_option("display.max_colwidth", 70)


## Steps

### 1. Create bounded synthetic observations


In [2]:
certificate_count = 96
certificates = pd.DataFrame({
    "certificate_id": [f"cert-{index:04d}" for index in range(certificate_count)],
    "fingerprint_group": rng.choice([f"fp-{index:02d}" for index in range(18)], certificate_count),
    "issuer": rng.choice(["CA-One", "CA-Two", "CA-Three"], certificate_count),
    "san_count": rng.integers(1, 18, certificate_count),
    "validity_days": rng.choice([30, 90, 180, 365], certificate_count, p=[0.10, 0.55, 0.10, 0.25]),
    "domain_age_days": rng.integers(1, 1500, certificate_count),
})
certificates["new_domain"] = (certificates["domain_age_days"] < 30).astype(int)
print(certificates.head(6).to_string(index=False))


certificate_id fingerprint_group   issuer  san_count  validity_days  domain_age_days  new_domain
     cert-0000             fp-09 CA-Three         13            365               56           0
     cert-0001             fp-03   CA-Two          1            180              958           0
     cert-0002             fp-16   CA-Two          2             90             1464           0
     cert-0003             fp-12   CA-One         12             90              612           0
     cert-0004             fp-12   CA-One          2             30              674           0
     cert-0005             fp-12   CA-Two         14             90              966           0


### 2. Analyze and rank the observations


In [3]:
fingerprint_summary = (
    certificates.groupby("fingerprint_group", as_index=False)
    .agg(certificates=("certificate_id", "count"), issuers=("issuer", "nunique"), median_sans=("san_count", "median"), new_domains=("new_domain", "sum"))
)
fingerprint_summary["pattern_score"] = (
    0.50 * np.minimum(fingerprint_summary["certificates"] / 8, 1)
    + 0.30 * np.minimum(fingerprint_summary["new_domains"] / 3, 1)
    + 0.20 * np.minimum(fingerprint_summary["median_sans"] / 12, 1)
).round(3)
ranked_fingerprints = fingerprint_summary.sort_values("pattern_score", ascending=False)
print(ranked_fingerprints.head(8).to_string(index=False))


fingerprint_group  certificates  issuers  median_sans  new_domains  pattern_score
            fp-12            10        3         10.0            1          0.767
            fp-02             8        3         10.5            0          0.675
            fp-04             6        3         12.5            1          0.675
            fp-05             9        3         10.0            0          0.667
            fp-03             7        3         11.0            0          0.621
            fp-08             8        2          7.0            0          0.617
            fp-17             6        2         12.0            0          0.575
            fp-11             5        2          7.0            1          0.529


## Checks

Run deterministic integrity and reasonableness checks.


In [4]:
assert certificates["certificate_id"].is_unique
assert ranked_fingerprints["pattern_score"].between(0, 1).all()
assert ranked_fingerprints["certificates"].sum() == certificate_count
print("Checks passed; certificate reuse is contextual evidence, not proof of control.")


Checks passed; certificate reuse is contextual evidence, not proof of control.


## Next Steps

- Compare first-seen dates across transparent certificate sources.
- Separate CDN and managed-platform reuse before review.
